### Phase 1: Financial Aggregation

> The Board asks: "How much money have we actually lost? Is it ₹10 Crores or ₹50 Crores?"

What you need to know: Before you can answer "how bad is the crisis?" you need to aggregate data. Aggregation means collapsing many rows into one summary number.

In [2]:
## Import dependencies and create Spark session

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 11:54:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 11:54:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Identify Dataset

- How much money have we actually lost?
- We need data that represents money and status.
- This matches with loan_amount (monry) and loan_status (loss condition) from the loans.csv dataset.

> Identified dataset: *loans.csv*

In [3]:
## Load the identified dataset and create a temporary view for SQL queries
loans = spark.read.csv("../Datasets/loans.csv", header=True)
loans.createOrReplaceTempView("loans")
loans.show()

26/04/21 11:54:11 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: ../Datasets/loans.csv.
java.io.FileNotFoundException: File ../Datasets/loans.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.sql.catalyst.analysis.Res

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/siddhu/Desktop/Databricks-EduFin-Investigation/Datasets/loans.csv. SQLSTATE: 42K03

Data Understanding

In [ ]:
query = """
SELECT DISTINCT loan_status FROM loans;
"""
spark.sql(query).show()

+-----------+
|loan_status|
+-----------+
|  Defaulted|
|    Overdue|
|     Active|
|     Closed|
+-----------+



> What each loan status represents?

- `Active:` Loan is ongoing → You took a loan and are paying monthly
- `Closed:` Loan fully completed → Loan finished after all EMIs
- `Overdue:` Payment is late → Missed last month’s EMI (Still recoverable/risk is increasing)
- `Defaulted:` Loan is failed → Stopped paying completely (Actual loss)

Phase Focus:

Combine COUNT, SUM, CASE WHEN, ROUND, and WHERE into one clean executive-ready query that answers: how many loans, how much disbursed, how many defaulted, and what is the default rate %?

In [11]:
query = """
SELECT
    COUNT(*) AS total_loans,
    ROUND(SUM(loan_amount) / 10000000.0, 2) AS total_disbursed_crores,
    COUNT(CASE WHEN loan_status = 'Defaulted' THEN 1 END) AS defaulted_loans,
    ROUND(
        SUM(CASE WHEN loan_status = 'Defaulted' THEN loan_amount END) / 10000000.0, 
        2
    ) AS total_loss_crores,
    ROUND(
        COUNT(CASE WHEN loan_status = 'Defaulted' THEN 1 END) * 100.0 / COUNT(*), 
        2
    ) AS default_rate_pct
FROM loans
WHERE loan_amount IS NOT NULL;
"""

In [12]:
risk_tiers = spark.sql(query)
risk_tiers.show()

+-----------+----------------------+---------------+-----------------+----------------+
|total_loans|total_disbursed_crores|defaulted_loans|total_loss_crores|default_rate_pct|
+-----------+----------------------+---------------+-----------------+----------------+
|       5000|                204.82|            590|            24.24|           11.80|
+-----------+----------------------+---------------+-----------------+----------------+

